# API Calls in Python

Learn how to make HTTP requests to external APIs, handle responses, and build robust API communication in your applications.

## What is an API?

An **API (Application Programming Interface)** is a set of rules for communication between applications. A **REST API** uses HTTP methods to perform operations on resources.

### HTTP Methods:
| Method | Purpose | Use Case |
|--------|---------|----------|
| **GET** | Retrieve data | Fetch data from server |
| **POST** | Create new data | Submit forms, create resources |
| **PUT** | Update data | Full replacement of resource |
| **PATCH** | Partial update | Update specific fields |
| **DELETE** | Remove data | Delete resources |

### API Response:
```json
{
  "status": 200,
  "data": { "id": 1, "name": "John" },
  "message": "Success"
}
```

### Common Status Codes:
- `200`: OK (Success)
- `201`: Created (Resource created)
- `400`: Bad Request (Invalid input)
- `401`: Unauthorized (Authentication needed)
- `404`: Not Found (Resource doesn't exist)
- `500`: Server Error (Server issue)

## Example 1: Basic GET Request

In [ ]:
import requests

# Simple GET request to a public API
url = "https://jsonplaceholder.typicode.com/posts/1"

response = requests.get(url)

# Check status code
print(f"Status Code: {response.status_code}")

# Parse JSON response
data = response.json()
print(f"\nTitle: {data['title']}")
print(f"Body: {data['body'][:100]}...")
print(f"\nFull response headers:")
for key, value in response.headers.items():
    if key in ['Content-Type', 'Content-Length', 'Server']:
        print(f"  {key}: {value}")

## Example 2: POST Request (Create Data)

In [ ]:
import requests
import json

url = "https://jsonplaceholder.typicode.com/posts"

# Data to send
payload = {
    "title": "My New Post",
    "body": "This is the content of my post",
    "userId": 1
}

# Headers (optional, requests auto-detects JSON)
headers = {
    "Content-Type": "application/json"
}

# POST request
response = requests.post(url, json=payload, headers=headers)

print(f"Status Code: {response.status_code}")
print(f"\nCreated Post:")
created_post = response.json()
print(json.dumps(created_post, indent=2))

## Example 3: Error Handling and Status Codes

In [ ]:
import requests
from requests.exceptions import RequestException, Timeout, ConnectionError

def safe_api_call(url, method="GET", **kwargs):
    """Make API call with comprehensive error handling"""
    try:
        # Timeout after 5 seconds
        response = requests.request(method, url, timeout=5, **kwargs)
        
        # Raise exception for HTTP errors
        response.raise_for_status()
        
        return {"success": True, "data": response.json(), "status": response.status_code}
    
    except Timeout:
        return {"success": False, "error": "Request timed out"}
    
    except ConnectionError:
        return {"success": False, "error": "Failed to connect to server"}
    
    except requests.HTTPError as e:
        return {"success": False, "error": f"HTTP Error {e.response.status_code}: {e.response.reason}"}
    
    except requests.JSONDecodeError:
        return {"success": False, "error": "Invalid JSON response"}
    
    except RequestException as e:
        return {"success": False, "error": f"Request failed: {str(e)}"}

# Test with valid request
print("Valid request:")
result = safe_api_call("https://jsonplaceholder.typicode.com/users/1")
print(f"Success: {result['success']}, Status: {result.get('status')}\n")

# Test with 404 error
print("Request with 404 error:")
result = safe_api_call("https://jsonplaceholder.typicode.com/users/99999")
print(f"Success: {result['success']}, Error: {result.get('error')}")

## Example 4: Authentication & Headers

In [ ]:
import requests
from requests.auth import HTTPBasicAuth
import os

# Example 1: Bearer Token (JWT)
headers = {
    "Authorization": "Bearer YOUR_API_KEY_HERE",
    "Content-Type": "application/json"
}

# Example 2: API Key in header
headers_with_key = {
    "X-API-Key": "your-api-key-123"
}

# Example 3: Basic Authentication
response = requests.get(
    "https://httpbin.org/basic-auth/user/pass",
    auth=HTTPBasicAuth('user', 'pass')
)
print(f"Basic Auth Status: {response.status_code}")

# Example 4: Custom headers with query parameters
url = "https://jsonplaceholder.typicode.com/posts"
params = {
    "userId": 1,
    "_limit": 2  # Limit results
}
headers = {
    "Accept": "application/json",
    "User-Agent": "MyApp/1.0"
}

response = requests.get(url, params=params, headers=headers)
print(f"\nQuery Parameters Status: {response.status_code}")
print(f"Retrieved {len(response.json())} posts")

# Example 5: Best practice - using environment variables for secrets
# api_key = os.getenv('API_KEY')  # Never hardcode secrets!
# headers = {"Authorization": f"Bearer {api_key}"}

## Example 5: Real-world - Handling Pagination

In [ ]:
import requests

def fetch_paginated_data(base_url, page_size=10, max_pages=3):
    """Fetch data across multiple pages"""
    all_data = []
    
    for page in range(1, max_pages + 1):
        # Adjust parameters based on API design
        params = {
            "_page": page,
            "_limit": page_size
        }
        
        try:
            response = requests.get(base_url, params=params, timeout=5)
            response.raise_for_status()
            
            data = response.json()
            
            # If no data returned, we've reached the end
            if not data:
                print(f"✓ Reached end of data at page {page}")
                break
            
            all_data.extend(data)
            print(f"✓ Page {page}: Retrieved {len(data)} items")
            
        except requests.RequestException as e:
            print(f"✗ Error on page {page}: {e}")
            break
    
    return all_data

# Real example
print("Fetching paginated posts:\n")
url = "https://jsonplaceholder.typicode.com/posts"
posts = fetch_paginated_data(url, page_size=10, max_pages=3)
print(f"\nTotal posts retrieved: {len(posts)}")

## Best Practices & Key Takeaways

### ✅ Do's:
- **Always set timeouts**: `requests.get(url, timeout=5)` to prevent hanging
- **Check status codes**: Use `response.raise_for_status()` to catch HTTP errors
- **Handle exceptions**: Catch `ConnectionError`, `Timeout`, `JSONDecodeError`
- **Use environment variables** for API keys/secrets (never hardcode!)
- **Set proper headers**: Include `Content-Type`, `User-Agent`, `Accept`
- **Implement retry logic**: Handle temporary failures gracefully
- **Use sessions** for multiple requests: `session = requests.Session()`
- **Validate responses**: Check if response is JSON before parsing

### ❌ Don'ts:
- Don't ignore timeouts (can hang indefinitely)
- Don't hardcode credentials in code
- Don't assume APIs are always available
- Don't make unlimited concurrent requests (use ThreadPool!)
- Don't parse JSON without error handling
- Don't trust input/output without validation

### Common Libraries:
- **requests**: Simple HTTP library (most popular)
- **aiohttp**: Async HTTP for concurrent requests
- **httpx**: Modern replacement for requests
- **urllib**: Built-in, lower-level HTTP

### Tips:
- Test with public APIs like `https://jsonplaceholder.typicode.com`
- Use browser DevTools to inspect actual API calls
- Read API documentation for rate limits and authentication
- Use response headers for pagination info